# Scratch Model With 32k BPE Tokenizer

This notebook mirrors `qwen_tokenizer_scratch_model.ipynb`, but uses the custom `tokenizers/english_bpe_32768` tokenizer and separate `bpe32k_*` checkpoints. It does not touch the Qwen-tokenizer checkpoints.

## 1. Find The Repo Root And Use The Current Kernel

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

cwd = Path.cwd().resolve()
project_dir = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "step9_train_tiny_transformer.py").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not find the repo root. Start VS Code from this project folder.")

os.chdir(project_dir)
print(f"project_dir: {project_dir}")
print(f"kernel python: {sys.executable}")

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, "-u", script, *args]
    print(">", shlex.join(cmd))
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


project_dir: D:\project\llm
kernel python: d:\project\llm\.venv\Scripts\python.exe


## 2. Check CUDA And 32k BPE Tokenizer

In [2]:
import torch
from transformers import AutoTokenizer

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

tokenizer_path = "tokenizers/english_bpe_32768"
tok = AutoTokenizer.from_pretrained(tokenizer_path)
print("tokenizer:", tokenizer_path)
print("vocab size:", len(tok))

text = "Where is Hong Kong?"
ids = tok.encode(text, add_special_tokens=False)
print("text:", repr(text))
print("ids:", ids)
print("tokens:", tok.convert_ids_to_tokens(ids))
print("round trip:", tok.decode(ids, skip_special_tokens=False) == text)

d:\project\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA GeForce RTX 4080 Laptop GPU
tokenizer: tokenizers/english_bpe_32768
vocab size: 32768
text: 'Where is Hong Kong?'
ids: [8629, 314, 10778, 11139, 34]
tokens: ['Where', 'Ġis', 'ĠHong', 'ĠKong', '?']
round trip: True


## 3. Prepare Data If Needed

The full FineWeb-Edu text can stay on disk. For BPE32K pretraining, build token shards once at `data/fineweb_edu_bpe32k_uint16/manifest.json`, then training reads the `.bin` shards with memmap instead of loading the `.txt` into RAM.


In [ ]:

# FineWeb is already downloaded locally. Leave this commented to avoid overwriting it.
# run_py(
#     "step29_prepare_fineweb_edu.py",
#     "--max-chars",
#     "0",
# )

# Run once before BPE32K pretraining if this manifest does not exist yet:
# data/fineweb_edu_bpe32k_uint16/manifest.json
run_py(
    "step32_tokenize_pretrain_shards.py",
    "--tokenizer-name",
    "tokenizers/english_bpe_32768",
    "--out-dir",
    "data/fineweb_edu_bpe32k_uint16",
    "--resume",
    "--batch-lines",
    "16384",
    "--chunk-chars",
    "10000000",
    "--num-threads",
    "24",
    "--dtype",
    "uint16",
    "--max-chars",
    "0",
)

# Uncomment only when rebuilding Alpaca-cleaned SFT files.
# run_py("step30_prepare_alpaca_cleaned.py")


> 'd:\project\llm\.venv\Scripts\python.exe' -u step32_tokenize_pretrain_shards.py --tokenizer-name tokenizers/english_bpe_32768 --out-dir data/fineweb_edu_bpe32k_uint16 --resume --batch-lines 16384 --num-threads 24 --dtype uint16 --max-chars 0
resuming from char position: 2,906,792,000
resuming shard 00006: 34,442,322 tokens
chars: 2,900,000,000 | tokens: 636,185,071 | current shard: 00006
chars: 3,000,000,000 | tokens: 658,153,155 | current shard: 00006
chars: 3,100,000,000 | tokens: 680,186,350 | current shard: 00006
saved shard 00006: 100,000,000 tokens
chars: 3,200,000,000 | tokens: 702,155,628 | current shard: 00007
chars: 3,300,000,000 | tokens: 724,101,841 | current shard: 00007
chars: 3,400,000,000 | tokens: 745,995,399 | current shard: 00007
chars: 3,500,000,000 | tokens: 767,856,845 | current shard: 00007
chars: 3,600,000,000 | tokens: 789,728,621 | current shard: 00007
saved shard 00007: 100,000,000 tokens
chars: 3,700,000,000 | tokens: 811,601,026 | current shard: 00008
cha

KeyboardInterrupt: 

## 4. Train Or Refresh 32k BPE Tokenizer

Run this if `tokenizers/english_bpe_32768` is missing or if the tokenizer training data changed.

In [ ]:
run_py(
    "step31_train_english_hf_tokenizer.py",
    "--vocab-size",
    "32768",
    "--out",
    "tokenizers/english_bpe_32768",
)

## 5. Smoke Train With 32k BPE Tokenizer

Optional quick check that tokenizer loading, model shape, and CUDA training all work.

In [ ]:
# run_py(
#     "step9_train_tiny_transformer.py",
#     "--preset",
#     "bpe32k_tiny_pretrain",
#     "--device",
#     "auto",
#     "--max-chars",
#     "100000",
#     "--max-iters",
#     "2",
#     "--save-every",
#     "0",
#     "--checkpoint",
#     "checkpoints/bpe32k_smoke/tiny_transformer.pt",
#     "--log-file",
#     "results/bpe32k_smoke_loss.csv",
#     "--log-every",
#     "1",
# )

## 6. Pretrain With 32k BPE Tokenizer

This preset uses `data/fineweb_edu_bpe32k_uint16/manifest.json` and memmap shard loading. Loss is logged to `results/bpe32k_pretrain_loss.csv`.


In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "bpe32k_tiny_pretrain",
    "--device",
    "auto",
    "--resume",
    "--save-every",
    "10000",
    "--log-file",
    "results/bpe32k_pretrain_loss.csv",
    "--log-every",
    "20",
)

## 6.1 Plot Pretrain Loss Log

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

log_path = Path("results/bpe32k_pretrain_loss.csv")
if not log_path.exists():
    print(f"loss log not found yet: {log_path}")
else:
    df = pd.read_csv(log_path)
    display(df.tail(10))

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(df["step"], df["loss"], label="loss", color="tab:blue")
    ax1.set_xlabel("step")
    ax1.set_ylabel("loss", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax1.grid(True, alpha=0.25)

    if "learning_rate" in df.columns:
        ax2 = ax1.twinx()
        ax2.plot(df["step"], df["learning_rate"], label="learning rate", color="tab:orange", alpha=0.75)
        ax2.set_ylabel("learning rate", color="tab:orange")
        ax2.tick_params(axis="y", labelcolor="tab:orange")

    plt.title("BPE 32k Pretrain Loss")
    fig.tight_layout()
    plt.show()

## 7. SFT With 32k BPE Tokenizer

Loss is logged to `results/bpe32k_sft_loss.csv`.

In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "bpe32k_tiny_sft",
    "--device",
    "auto",
    "--resume",
    "--save-every",
    "1000",
    "--log-file",
    "results/bpe32k_sft_loss.csv",
    "--log-every",
    "20",
)

## 7.1 Plot SFT Loss Log


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

log_path = Path("results/bpe32k_sft_loss.csv")
if not log_path.exists():
    print(f"loss log not found yet: {log_path}")
else:
    df = pd.read_csv(log_path)
    display(df.tail(10))

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(df["step"], df["loss"], label="loss", color="tab:blue")
    ax1.set_xlabel("step")
    ax1.set_ylabel("loss", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax1.grid(True, alpha=0.25)

    if "learning_rate" in df.columns:
        ax2 = ax1.twinx()
        ax2.plot(df["step"], df["learning_rate"], label="learning rate", color="tab:orange", alpha=0.75)
        ax2.set_ylabel("learning rate", color="tab:orange")
        ax2.tick_params(axis="y", labelcolor="tab:orange")

    plt.title("BPE 32k SFT Loss")
    fig.tight_layout()
    plt.show()


## 8. Generate From 32k BPE Scratch Model

In [ ]:
run_py(
    "step10_generate.py",
    "--preset",
    "bpe32k_tiny_pretrain",
    "--device",
    "auto",
    "--prompt",
    "Hong Kong is",
    "--max-new-tokens",
    "80",
    "--temperature",
    "0.3",
)

In [ ]:
# After SFT, use this instead.
# run_py(
#     "step10_generate.py",
#     "--preset",
#     "bpe32k_tiny_sft",
#     "--device",
#     "auto",
#     "--prompt",
#     "How are you?",
#     "--max-new-tokens",
#     "80",
#     "--temperature",
#     "0.3",
# )